# 00 — Exploración de Arquitecturas GenAI y Prompt Engineering

**Proyecto:** DMC Institute — Clasificación de leads y recomendación de cursos  
**Modelo:** Qwen2.5-7B-Instruct (pre-entrenado, sin fine-tuning)  
**Entorno recomendado:** Google Colab T4 (16GB VRAM)

---

**Estructura del notebook:**
1. Setup e instalación
2. Overview de arquitecturas de IA Generativa
3. Carga del modelo pre-entrenado (HuggingFace `pipeline` API)
4. Experimentos de prompt engineering (zero-shot, few-shot, system prompt, CoT)
5. Comparativa de resultados y conclusiones

---
## 📦 Task 1: Setup e Instalación

In [4]:
# Celda 0: Instalación
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "transformers>=4.40.0",
    "torch>=2.2.0",
    "accelerate>=0.28.0",
    "bitsandbytes>=0.43.0",
    "sentencepiece",
])
print("✅ Dependencias instaladas")

✅ Dependencias instaladas


In [1]:
!nvidia-smi

Sun Jun  7 22:46:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# Celda 1: Verificar entorno
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch: 2.11.0+cu128
CUDA disponible: True
GPU: Tesla T4
VRAM: 15.6 GB


---
## 🧠 Task 2: Overview de Arquitecturas GenAI

# 🧠 Arquitecturas de IA Generativa — Overview

## Transformer-based (LLMs)
Los modelos de lenguaje como GPT, Llama, Qwen usan la arquitectura Transformer con atención multi-cabeza.
- **Encoder-only** (BERT): clasificación, embeddings
- **Decoder-only** (GPT, Llama, Qwen): generación de texto autoregresiva ← usamos este
- **Encoder-Decoder** (T5, BART): traducción, resumen

## Diffusion Models
Aprenden a revertir un proceso de ruido gaussiano iterativo.
- **Denoising Diffusion Probabilistic Models (DDPM)**: base matemática
- **Latent Diffusion (Stable Diffusion, FLUX)**: operan en espacio latente comprimido ← Plan C

## ¿Por qué Qwen2.5-7B para DMC?
- 7B parámetros: balance entre capacidad y costo de inferencia en T4 (16GB VRAM)
- Instrucción-tuned: responde bien a prompts de sistema estructurados
- Multilingüe con fuerte soporte en español

In [5]:
# Celda 3: Visualizar el flujo de generación autoregresiva
diagram = """
GENERACIÓN AUTOREGRESIVA (Transformer Decoder)
================================================

Input tokens: ["Quiero", "aprender", "Power"]
       ↓
  [Embedding Layer]
       ↓
  [Self-Attention]  ← cada token "atiende" a todos los anteriores
       ↓
  [Feed-Forward]
       ↓
  [Softmax sobre vocabulario]
       ↓
Output token: "BI"  → se agrega al input → repite hasta <EOS>

Para DMC: el modelo genera la clasificación del lead token por token
Input:  "Mi empresa me pidió certificarme en Azure"
Output: {"motivation": "company_requirement", "score": "hot", ...}
"""
print(diagram)


GENERACIÓN AUTOREGRESIVA (Transformer Decoder)

Input tokens: ["Quiero", "aprender", "Power"]
       ↓
  [Embedding Layer]
       ↓
  [Self-Attention]  ← cada token "atiende" a todos los anteriores
       ↓
  [Feed-Forward]
       ↓
  [Softmax sobre vocabulario]
       ↓
Output token: "BI"  → se agrega al input → repite hasta <EOS>

Para DMC: el modelo genera la clasificación del lead token por token
Input:  "Mi empresa me pidió certificarme en Azure"
Output: {"motivation": "company_requirement", "score": "hot", ...}



---
## 🤗 Task 3: Cargar Modelo Pre-entrenado de HuggingFace

In [6]:
# Celda 4: Cargar modelo pre-entrenado (sin fine-tuning)
from transformers import pipeline
import torch

# Qwen2.5-7B-Instruct — modelo base que luego fine-tunearemos en el Plan B
generator = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-7B-Instruct",
    torch_dtype=torch.float16,
    device_map="auto",
)
print("✅ Modelo cargado: Qwen2.5-7B-Instruct (pre-entrenado, sin fine-tuning)")
print(f"Parámetros: ~7B")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

✅ Modelo cargado: Qwen2.5-7B-Instruct (pre-entrenado, sin fine-tuning)
Parámetros: ~7B


In [7]:
# Celda 5: Helper para generación limpia
def generate(messages: list, max_tokens: int = 300) -> str:
    """Genera texto dado un historial de mensajes en formato chat."""
    output = generator(
        messages,
        max_new_tokens=max_tokens,
        do_sample=False,          # greedy decoding para reproducibilidad
        return_full_text=False,
    )
    return output[0]["generated_text"].strip()

# Prueba rápida
response = generate([
    {"role": "user", "content": "¿Qué es Power BI en una oración?"}
])
print(f"Respuesta: {response}")

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Respuesta: Power BI es un servicio de análisis y visualización de datos desarrollado por Microsoft que permite a los usuarios transformar y conectar datos de diversas fuentes para crear informes y visualizaciones interactivas.


---
## 🧪 Task 4: Experimentos de Prompt Engineering

In [8]:
# Celda 6: Zero-shot prompting
# Le pedimos la tarea sin ejemplos

TAREA = "Clasifica la motivación de este mensaje de un potencial estudiante de DMC Institute: '{msg}'. Responde con: growth / salary / company_requirement / academic / undefined"

test_messages = [
    "Mi empresa me pidió que me certifique en Azure.",
    "Quiero ganar más dinero cambiando a data science.",
    "Me interesa aprender machine learning por curiosidad.",
]

print("=== ZERO-SHOT ===")
for msg in test_messages:
    prompt = TAREA.format(msg=msg)
    response = generate([{"role": "user", "content": prompt}], max_tokens=50)
    print(f"Input:  {msg}")
    print(f"Output: {response}")
    print()

[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== ZERO-SHOT ===


[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Input:  Mi empresa me pidió que me certifique en Azure.
Output: company_requirement



[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Input:  Quiero ganar más dinero cambiando a data science.
Output: salary

Input:  Me interesa aprender machine learning por curiosidad.
Output: academic



In [9]:
# Celda 7: Few-shot prompting
# Damos 3 ejemplos antes de la pregunta real

FEW_SHOT_PROMPT = """Clasifica la motivación del mensaje. Solo responde con una palabra.

Ejemplos:
- "Quiero subir mi sueldo aprendiendo datos" → salary
- "Mi jefe me pidió certificarme" → company_requirement
- "Me gusta la IA y quiero aprender más" → academic
- "Quiero cambiar de carrera a data analyst" → growth

Ahora clasifica: "{msg}"
Respuesta:"""

print("=== FEW-SHOT ===")
for msg in test_messages:
    prompt = FEW_SHOT_PROMPT.format(msg=msg)
    response = generate([{"role": "user", "content": prompt}], max_tokens=20)
    print(f"Input:  {msg}")
    print(f"Output: {response}")
    print()

[transformers] Both `max_new_tokens` (=20) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== FEW-SHOT ===


[transformers] Both `max_new_tokens` (=20) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Input:  Mi empresa me pidió que me certifique en Azure.
Output: company_requirement



[transformers] Both `max_new_tokens` (=20) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Input:  Quiero ganar más dinero cambiando a data science.
Output: salary

Input:  Me interesa aprender machine learning por curiosidad.
Output: academic



In [10]:
# Celda 8: System prompt con rol y formato JSON
SYSTEM = """Eres un clasificador de leads para DMC Institute.
Dado un mensaje, responde SOLO con JSON:
{"motivation": "growth|salary|company_requirement|academic|undefined", "score": "hot|warm|cold"}

Reglas de score:
- hot: expresa intención de compra o urgencia alta
- warm: motivación clara pero sin decisión de compra
- cold: exploración vaga o sin urgencia"""

print("=== SYSTEM PROMPT + JSON ===")
for msg in test_messages:
    response = generate([
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": msg},
    ], max_tokens=80)
    print(f"Input:  {msg}")
    print(f"Output: {response}")
    print()

[transformers] Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== SYSTEM PROMPT + JSON ===


[transformers] Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Input:  Mi empresa me pidió que me certifique en Azure.
Output: {"motivation": "academic", "score": "warm"}



[transformers] Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Input:  Quiero ganar más dinero cambiando a data science.
Output: {"motivation": "salary", "score": "warm"}

Input:  Me interesa aprender machine learning por curiosidad.
Output: {"motivation": "academic", "score": "cold"}



In [11]:
# Celda 9: Chain-of-Thought — pedimos razonamiento explícito antes de la respuesta
COT_SYSTEM = """Eres un clasificador de leads para DMC Institute.
Antes de clasificar, razona brevemente (1 oración) sobre la motivación del usuario.
Luego responde con JSON: {"motivation": "...", "score": "..."}

Formato de respuesta:
Razonamiento: <1 oración>
JSON: {"motivation": "...", "score": "..."}"""

print("=== CHAIN-OF-THOUGHT ===")
for msg in test_messages:
    response = generate([
        {"role": "system", "content": COT_SYSTEM},
        {"role": "user", "content": msg},
    ], max_tokens=120)
    print(f"Input:  {msg}")
    print(f"Output: {response}")
    print()

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== CHAIN-OF-THOUGHT ===


[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Input:  Mi empresa me pidió que me certifique en Azure.
Output: Razonamiento: El usuario busca información sobre el proceso de certificación en Azure para cumplir con la solicitud de su empresa.
JSON: {"motivation": "Cumplimiento de requisitos laborales", "score": "8"}



[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Input:  Quiero ganar más dinero cambiando a data science.
Output: Razonamiento: El usuario busca una oportunidad para aumentar su ingreso potencial mediante una transición hacia una carrera en data science.
JSON: {"motivation": "cambio de carrera para aumentar ingresos", "score": "8"}

Input:  Me interesa aprender machine learning por curiosidad.
Output: Razonamiento: El usuario busca información sobre machine learning por interés personal y curiosidad.
JSON: {"motivation": "personal interest and curiosity", "score": "3"}



---
## 📊 Task 5: Comparativa de Estrategias y Conclusiones

In [12]:
# Celda 10: Comparativa de las 4 estrategias de prompting

import json

GROUND_TRUTH = ["company_requirement", "salary", "academic"]

results_table = []

for msg, gt in zip(test_messages, GROUND_TRUTH):
    row = {"Mensaje": msg[:40] + "...", "Ground Truth": gt}
    
    # Zero-shot
    r = generate([{"role": "user", "content": TAREA.format(msg=msg)}], max_tokens=20)
    row["Zero-shot"] = r.strip().lower()[:30]
    
    # Few-shot
    r = generate([{"role": "user", "content": FEW_SHOT_PROMPT.format(msg=msg)}], max_tokens=20)
    row["Few-shot"] = r.strip().lower()[:30]
    
    # System prompt JSON
    r = generate([{"role": "system", "content": SYSTEM}, {"role": "user", "content": msg}], max_tokens=60)
    try:
        parsed = json.loads(r.strip())
        row["System+JSON"] = parsed.get("motivation", "error")
    except:
        row["System+JSON"] = "json_error"
    
    # CoT
    r = generate([{"role": "system", "content": COT_SYSTEM}, {"role": "user", "content": msg}], max_tokens=100)
    try:
        json_part = r.split("JSON:")[-1].strip() if "JSON:" in r else r
        parsed = json.loads(json_part)
        row["CoT"] = parsed.get("motivation", "error")
    except:
        row["CoT"] = "json_error"
    
    results_table.append(row)

# Imprimir tabla
print(f"{'Mensaje':<45} {'GT':<25} {'Zero':<25} {'Few':<25} {'Sys+JSON':<25} {'CoT':<25}")
print("-" * 145)
for row in results_table:
    print(f"{row['Mensaje']:<45} {row['Ground Truth']:<25} {row['Zero-shot']:<25} {row['Few-shot']:<25} {row['System+JSON']:<25} {row['CoT']:<25}")

[transformers] Both `max_new_tokens` (=20) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=20) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/doc

Mensaje                                       GT                        Zero                      Few                       Sys+JSON                  CoT                      
-------------------------------------------------------------------------------------------------------------------------------------------------
Mi empresa me pidió que me certifique en...   company_requirement       company_requirement       company_requirement       academic                  Cumplimiento de requisitos laborales
Quiero ganar más dinero cambiando a data...   salary                    salary                    salary                    salary                    cambio de carrera para aumentar ingresos
Me interesa aprender machine learning po...   academic                  academic                  academic                  academic                  personal interest and curiosity


## 📊 Conclusiones del Experimento de Prompt Engineering

### Hallazgos principales

| Estrategia | Resultado observado | Control del formato | Observación |
|---|---|---|---|
| Zero-shot | Clasificó correctamente los ejemplos simples | Bajo | El modelo respondió con la etiqueta esperada, pero sin estructura JSON ni garantías de consistencia |
| Few-shot | Clasificó correctamente los ejemplos simples | Medio | Los ejemplos ayudaron a mantener las etiquetas esperadas, aunque la salida sigue dependiendo mucho del prompt |
| System prompt + JSON | Generó JSON válido, pero con errores de clasificación | Alto | Fue la mejor estrategia para controlar la estructura de salida, aunque confundió una motivación importante |
| Chain-of-Thought | Produjo razonamientos útiles, pero etiquetas no estandarizadas | Bajo/Medio | Explica mejor su decisión, pero genera respuestas más largas, variables y difíciles de parsear automáticamente |

### Interpretación

El modelo pre-entrenado mostró buen desempeño en ejemplos simples usando `zero-shot` y `few-shot`, pero estos resultados no son suficientes para concluir que el sistema sea confiable en producción. La muestra evaluada es pequeña y no cubre la variedad real de mensajes que podrían enviar los leads de DMC.

La estrategia `System prompt + JSON` fue la más adecuada como base técnica porque permite obtener una salida estructurada. Sin embargo, todavía puede cometer errores de clasificación, como confundir una motivación de tipo `company_requirement` con `academic`.

Por otro lado, `Chain-of-Thought` ayuda a obtener explicaciones más detalladas, pero reduce el control sobre el formato final. En varios casos el modelo generó motivaciones en lenguaje natural y puntajes numéricos, en lugar de respetar las etiquetas esperadas.

### Conclusión general

El prompt engineering permite construir una primera línea base para clasificar leads, pero **no es suficiente por sí solo para un sistema robusto y automatizable**.

Para un caso real de DMC, la mejor base es usar un `system prompt` con salida en JSON, complementado con validación, post-procesamiento y eventualmente fine-tuning.

Esto permitiría:

1. Garantizar que la respuesta tenga un formato JSON válido.
2. Forzar el uso de etiquetas controladas como `company_requirement`, `salary` o `academic`.
3. Reducir respuestas ambiguas o demasiado abiertas.
4. Mejorar la clasificación de motivaciones específicas del negocio.
5. Evitar errores en campos críticos como `motivation` y `score`.

En resumen, el modelo pre-entrenado demuestra capacidad para entender la intención general del lead, pero requiere mecanismos adicionales de control para ser confiable en un flujo comercial real.